# Model 3: BIST30 Stock Intraday Reaction Forecaster

## Executive Summary & Modeling Scope

The **BIST30 Stock Intraday Reaction Forecaster** predicts execution-aware intraday stock return percentages for BIST30 equities across three key post-opening reaction windows:
- **Window 2 (`first_reaction`, 10:30 – 11:30 TRT)**: Immediate momentum continuation or reversal after the opening session.
- **Window 3 (`midday_followup`, 11:30 – 14:30 TRT)**: Midday institutional accumulation and trend development.
- **Window 5 (`closing_session`, 16:00 – 18:15 TRT)**: End-of-day resolution and closing auction dynamics.

### Target Formulation
The regression target is defined as the execution-aware return %:
$$\text{Target Return}\% = \frac{\text{Window VWAP} - P_{\text{W1 reference}}}{P_{\text{W1 reference}}} \times 100$$

where $P_{\text{W1 reference}}$ is BofA's opening Buy VWAP (fallback: Market W1 VWAP).

> Zero lookahead guarantee: All input features use data available up to **10:30 TRT (end of Window 1)** on session $T$.

In [1]:
import duckdb
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

from mdk_trading_oracle.core.db import DuckDBManager
from mdk_trading_oracle.core.config import get_settings
from mdk_trading_oracle.models.stock_reaction.features import StockReactionFeatureExtractor
from mdk_trading_oracle.models.stock_reaction.forecaster import StockReactionForecaster, StockReactionModelArena
from mdk_trading_oracle.models.stock_reaction.orchestrator import StockReactionOrchestrator
from mdk_trading_oracle.models.stock_reaction.models import (
    ReturnDirectionClassifier,
    ReturnThresholdProfile,
    StockReactionBayesianModel,
    StockReactionLightGBMModel,
    StockReactionNaivePersistenceModel,
    StockReactionRollingMeanModel,
    StockReactionXGBoostModel,
)

settings = get_settings()
db = DuckDBManager(read_only=True)
print(f"[OK] DuckDB Database: {settings.database_path}")
print(f"[OK] Stock Reaction Engine Ready")


[OK] DuckDB Database: /Users/ozkanyildirim/data/mdk_oracle/database/mdk_oracle.duckdb
[OK] Stock Reaction Engine Ready


## 1. Feature Extraction & Microstructure Clusters

Extract 8 quantitative feature clusters (47 features) for BIST30 equities (e.g. `AKBNK`).

In [2]:
extractor = StockReactionFeatureExtractor(symbol="AKBNK", db=db)
df_feats = extractor.extract_features()
feat_cols = extractor.get_feature_columns()

print(f"[OK] Extracted {len(df_feats)} sessions for AKBNK across {len(feat_cols)} engineered feature columns.")
display(df_feats.select([
    "trade_date", "symbol", "feat_bofa_w1_net_flow_tl", "feat_bofa_w1_vol_share",
    "feat_comp_w1_net_flow_tl", "feat_w1_bofa_tra_contra_signal",
    "feat_bofa_t1_open_qty", "target_w2_return_pct", "target_w5_return_pct"
]).head(8).to_pandas())


[09/06/26 14:17:03] INFO     [AKBNK] Extracted 21 rows × 66 columns (feature clusters: 8, lookback: 12m)

[OK] Extracted 21 sessions for AKBNK across 58 engineered feature columns.


,trade_date,symbol,feat_bofa_w1_net_flow_tl,feat_bofa_w1_vol_share,feat_comp_w1_net_flow_tl,feat_w1_bofa_tra_contra_signal,feat_bofa_t1_open_qty,target_w2_return_pct,target_w5_return_pct
0,2026-03-02,AKBNK,6.405484e+07,1.0,-2.317188e+08,1.0,NaN,-0.698071,-1.228329
1,2026-03-03,AKBNK,5.184623e+07,1.0,-1.138573e+08,0.0,12029186.0,0.365728,-2.768442
2,2026-03-04,AKBNK,2.383818e+07,1.0,3.059595e+07,0.0,22563719.0,0.549132,0.090364
3,2026-03-05,AKBNK,9.089762e+07,1.0,-1.937698e+07,1.0,20639425.0,-0.212256,-1.088185
4,2026-03-06,AKBNK,3.338039e+07,1.0,4.786558e+07,0.0,19843384.0,0.469681,-4.238933
5,2026-03-09,AKBNK,1.009651e+08,1.0,-2.559280e+07,0.0,12175213.0,-1.633655,1.363529
6,2026-03-10,AKBNK,-2.354004e+08,1.0,2.764183e+08,-1.0,8328617.0,-0.962431,0.155544
7,2026-03-11,AKBNK,-4.407109e+07,1.0,2.074757e+08,-1.0,5175846.0,-0.312817,0.111812


## 2. Empirical Quantile Thresholds Profile

Inspect empirical return distributions ($P_{25}, P_{50}, P_{85}$) computed in `silver_stock_reaction_thresholds`.

In [3]:
conn = db.get_connection()
df_thresh = conn.execute("""
    SELECT symbol, window_name, up_p25_pct, up_p50_pct, up_p85_pct, down_p85_pct, total_sessions
    FROM silver_stock_reaction_thresholds
    WHERE symbol IN ('AKBNK', 'GARAN', 'THYAO', 'ASELS', 'KCHOL')
    ORDER BY symbol, window_name;
""").pl()
display(df_thresh.to_pandas())


,symbol,window_name,up_p25_pct,up_p50_pct,up_p85_pct,down_p85_pct,total_sessions
0,AKBNK,closing_session,0.331774,1.946365,3.311829,2.556830,20
1,AKBNK,first_reaction,0.333701,0.569880,0.784210,0.853662,21
2,AKBNK,midday_followup,0.445921,0.610587,0.951469,1.649870,21
3,ASELS,closing_session,1.472742,1.904848,3.548594,2.560775,20
4,ASELS,first_reaction,0.605202,0.678891,1.033177,1.362065,21
5,ASELS,midday_followup,1.145039,1.839588,1.888323,1.863949,21
6,GARAN,closing_session,1.091274,2.535549,2.577264,3.079040,20
7,GARAN,first_reaction,0.432010,0.809933,1.147315,0.930751,21
8,GARAN,midday_followup,0.268596,0.436276,0.907249,1.826672,21
9,KCHOL,closing_session,0.487753,1.042057,1.923315,2.918979,20


## 3. Candidate Model Arena & Tournament

Run expanding-window walk-forward validation across candidate paradigms for `AKBNK` (Window 2: `first_reaction`).

In [4]:
forecaster_w2 = StockReactionForecaster(symbol="AKBNK", window="w2", db=db)
X_train, y_train = forecaster_w2._prepare_training_data()
thresholds = forecaster_w2._load_thresholds()

arena = StockReactionModelArena(symbol="AKBNK", window="w2", thresholds=thresholds)
scoreboard, champion = arena.run_tournament(X_train, y_train, min_train_samples=5)

display(scoreboard[["Model", "hit_rate_pct", "picp_90_pct", "mae_pct", "rmse_pct", "sample_size"]])
print(f"Champion Model: {champion.model_name}")


[09/06/26 14:17:04] INFO     [AKBNK] Extracted 21 rows × 66 columns (feature clusters: 8, lookback: 12m)

[09/06/26 14:17:05] INFO     [AKBNK/w2] Champion: 'Bayesian Ridge Probabilistic' (Hit: 55.6% | Alpha vs Hurdle:    
                             +0.0% | PICP90: 66.7% | MAE: 2.005% | n=9)

,Model,hit_rate_pct,picp_90_pct,mae_pct,rmse_pct,sample_size
0,Hurdle 0: Naive Persistence,22.222222,55.555556,1.882790,2.465200,9
1,Hurdle 1: 5-Day Rolling Mean,55.555556,77.777778,1.378027,1.631397,9
2,Hurdle 2: Always Long (+1),44.444444,77.777778,1.053998,1.770583,9
3,Hurdle 3: Always Short (-1),55.555556,77.777778,1.545338,1.705439,9
4,Bayesian Ridge Probabilistic,55.555556,66.666667,2.005386,2.447725,9
5,LightGBM Non-Linear Ensemble,0.000000,0.000000,999.000000,999.000000,0
6,XGBoost Non-Linear Ensemble,0.000000,0.000000,999.000000,999.000000,0


Champion Model: Bayesian Ridge Probabilistic


## 4. Live Upcoming Session Signal Card ($T+1$)

Query the latest active forecasts from the Gold tables (`gold_bofa_stock_reaction_*_forecasts`).

In [5]:
orchestrator = StockReactionOrchestrator(db=db)
df_w2_live = orchestrator.get_latest_forecasts(window="w2")
if df_w2_live is not None and not df_w2_live.is_empty():
    display(df_w2_live.to_pandas().head(10))
else:
    print("[INFO] No live forecasts currently active. Run pipeline to generate live inference.")


[09/06/26 14:17:06] INFO     Symbol list resolved from Bronze layer: 30 stocks (active BIST30)

                    INFO     StockReactionOrchestrator initialized: 30 symbols × 3 windows (90 total forecaster    
                             runs)

,forecast_date,symbol,window_name,predicted_return_pct,predicted_return_lower_90,predicted_return_upper_90,predicted_direction,direction_confidence,predicted_playbook,bofa_w1_direction,bofa_w1_net_flow_tl,bofa_w1_volume_share,model_name,generated_at
0,2026-03-31,KRDMD,first_reaction,-7.9787,-10.7206,-5.2369,STRONG_DECLINE,1.00,LIQUIDITY_FADE,SELL,-8.841570e+06,1.0,Bayesian Ridge Probabilistic,2026-08-28 00:39:49.347072
1,2026-03-31,DSTKF,first_reaction,-3.5568,-6.1306,-0.9829,STRONG_DECLINE,0.35,NEUTRAL_WAIT,BUY,3.037981e+07,1.0,Bayesian Ridge Probabilistic,2026-08-28 00:39:19.481209
2,2026-03-31,HALKB,first_reaction,-2.4814,-6.4401,1.4773,STRONG_DECLINE,0.35,NEUTRAL_WAIT,SELL,-4.743467e+06,1.0,Bayesian Ridge Probabilistic,2026-08-25 00:22:30.637358
3,2026-03-31,TUPRS,first_reaction,-2.3789,-5.1848,0.4270,STRONG_DECLINE,0.35,NEUTRAL_WAIT,SELL,-1.893978e+08,1.0,Bayesian Ridge Probabilistic,2026-08-28 00:40:30.868466
4,2026-03-31,BIMAS,first_reaction,-2.2323,-2.9835,-1.4812,STRONG_DECLINE,0.35,NEUTRAL_WAIT,SELL,-3.174231e+07,1.0,Bayesian Ridge Probabilistic,2026-08-28 00:39:15.982379
5,2026-03-31,ODAS,first_reaction,-2.0646,-3.6591,-0.4701,STRONG_DECLINE,1.00,LIQUIDITY_FADE,SELL,-1.661834e+06,1.0,Bayesian Ridge Probabilistic,2026-08-25 00:22:46.667087
6,2026-03-31,TOASO,first_reaction,-1.5712,-4.2899,1.1475,STRONG_DECLINE,1.00,LIQUIDITY_FADE,SELL,-1.365692e+06,1.0,Bayesian Ridge Probabilistic,2026-08-28 00:40:20.935391
7,2026-03-31,OYAKC,first_reaction,-1.5598,-2.5735,-0.5462,STRONG_DECLINE,1.00,SECTOR_ROTATION,BUY,3.045344e+06,1.0,Bayesian Ridge Probabilistic,2026-08-25 00:22:49.823406
8,2026-03-31,TAVHL,first_reaction,-1.2921,-4.3672,1.7830,STRONG_DECLINE,1.00,LIQUIDITY_FADE,SELL,-9.926270e+06,1.0,Bayesian Ridge Probabilistic,2026-08-28 00:40:11.446999
9,2026-03-31,EKGYO,first_reaction,-1.2818,-4.7688,2.2053,STRONG_DECLINE,0.35,NEUTRAL_WAIT,SELL,-6.066292e+04,1.0,Bayesian Ridge Probabilistic,2026-08-28 00:39:23.223718


## 5. Performance Ledger & Backtest Verification

Inspect historical reconciled track records and walk-forward simulation ledgers in DuckDB.

In [6]:
conn = db.get_connection()
perf_w2 = conn.execute("""
    SELECT symbol, COUNT(*) as n, AVG(CAST(is_direction_hit AS INTEGER))*100 as hit_rate_pct,
           AVG(absolute_error_pct) as avg_mae_pct, AVG(CAST(is_inside_90_ci AS INTEGER))*100 as picp_90_pct
    FROM gold_bofa_stock_reaction_w2_performance
    WHERE actual_return_pct IS NOT NULL
    GROUP BY symbol
    ORDER BY hit_rate_pct DESC;
""").pl()
display(perf_w2.to_pandas())


,symbol,n,hit_rate_pct,avg_mae_pct,picp_90_pct
0,AEFES,1,100.0,0.443537,100.0
1,EKGYO,1,100.0,0.055713,100.0
2,EREGL,1,100.0,0.267579,100.0
3,GUBRF,1,100.0,0.017602,100.0
4,KRDMD,1,100.0,2.334870,100.0
5,THYAO,1,100.0,0.269900,100.0
6,TAVHL,1,100.0,0.632529,100.0
7,SISE,1,100.0,0.170059,100.0
8,ASELS,1,100.0,0.545270,100.0
9,TUPRS,1,100.0,2.371803,100.0


## 6. DuckDB Gold Layer Tables Overview

Verify the 9 Model 3 Gold tables.

In [7]:
tables = conn.execute("""
    SHOW TABLES;
""").fetchall()
stock_reaction_tables = [t[0] for t in tables if "stock_reaction" in t[0]]
print(f"Stock Reaction Gold & Silver Tables ({len(stock_reaction_tables)}):\n")
for tbl in sorted(stock_reaction_tables):
    cnt = conn.execute(f"SELECT COUNT(*) FROM {tbl};").fetchone()[0]
    print(f"  - {tbl:<45}: {cnt:>6,} rows")


Stock Reaction Gold & Silver Tables (10):

  - gold_bofa_stock_reaction_w2_backtests        :    432 rows
  - gold_bofa_stock_reaction_w2_forecasts        :     35 rows
  - gold_bofa_stock_reaction_w2_performance      :     30 rows
  - gold_bofa_stock_reaction_w3_backtests        :    432 rows
  - gold_bofa_stock_reaction_w3_forecasts        :     35 rows
  - gold_bofa_stock_reaction_w3_performance      :     30 rows
  - gold_bofa_stock_reaction_w5_backtests        :    405 rows
  - gold_bofa_stock_reaction_w5_forecasts        :     35 rows
  - gold_bofa_stock_reaction_w5_performance      :     30 rows
  - silver_stock_reaction_thresholds             :    135 rows
